# SpliceTransformer Splice Site Prediction Examples

This notebook demonstrates tissue-specific splice site prediction using SpliceTransformer:
- **Basic splice site prediction** -- Predict splice sites with synthetic sequences
- **Interpreting predictions** -- Splice type channels and tissue channels
- **Finding high-confidence splice sites** -- Threshold-based site detection
- **Tissue-specific analysis** -- Compare splicing across 15 human tissues

## 📥 Imports

## 📦 Shared Data Types

### `SpliceTransformerType` (Enum)
Model type to use for prediction.

| Member | Value | Description |
|--------|-------|-------------|
| `NEITHER` | `0` | Not a splice site |
| `ACCEPTOR` | `1` | Acceptor splice site |
| `DONOR` | `2` | Donor splice site |

### Tissue Channel Mapping
Tissue-specific prediction channels in `prediction[:, :, channel_idx]`.

| Tissue | Channel Index |
|--------|---------------|
| `AVERAGE` | `3:` (slice over all tissue channels) |
| `ADIPOSE_TISSUE` | `3` |
| `BLOOD` | `4` |
| `BLOOD_VESSEL` | `5` |
| `BRAIN` | `6` |
| `COLON` | `7` |
| `HEART` | `8` |
| `KIDNEY` | `9` |
| `LIVER` | `10` |
| `LUNG` | `11` |
| `MUSCLE` | `12` |
| `NERVE` | `13` |
| `SMALL_INTESTINE` | `14` |
| `SKIN` | `15` |
| `SPLEEN` | `16` |
| `STOMACH` | `17` |


In [1]:
import numpy as np

from proto_tools import (
    SPLICE_TISSUE_CHANNEL_INDEX,
    SpliceTransformerConfig,
    SpliceTransformerInput,
    SpliceTransformerType,
    run_splice_transformer,
)


## 🧬 1. Basic Splice Site Prediction

Predict splice sites in a synthetic sequence with flanking context.
The model requires:
- Target sequence (typically 1000bp)
- Left context (typically 4000bp)
- Right context (typically 4000bp)

### API Reference

**`SpliceTransformerInput`**

| Field | Type | Description |
|-------|------|-------------|
| `target_seqs` | `List[str]` | Sequence(s) on which to make splicing predictions |
| `left_contexts` | `List[str]` | Sequence(s) of the left (5') context, must match length of `target_seqs` |
| `right_contexts` | `List[str]` | Sequence(s) of the right (3') context, must match length of `target_seqs` |

**`SpliceTransformerConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `context_length` | `int` | `4000` | Context length on both left and right of target sequence |

**`SpliceTransformerOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `prediction` | `list[list[list[float]]]` | Prediction tensor of shape `[batch, target_length, 18]`. Channels 0-2: splice type (neither, acceptor, donor). Channels 3-17: tissue-specific usage (15 tissues). Convert to numpy with `np.array(result.prediction)` for slicing. |

In [ ]:
# Create synthetic sequences
target_seq = "ATGC" * 250  # 1000bp target
left_ctx = "CGTA" * 1000   # 4000bp left context
right_ctx = "TACG" * 1000  # 4000bp right context

inputs = SpliceTransformerInput(
    target_seqs=[target_seq],
    left_contexts=[left_ctx],
    right_contexts=[right_ctx],
)
config = SpliceTransformerConfig(
    context_length=4000,
    device="cpu",  # Use "cuda" if GPU is available
    verbose=True,
)

result = run_splice_transformer(inputs, config)
pred = np.array(result.prediction)
print(f"Prediction shape: {pred.shape}")
print(f"  batch={pred.shape[0]}, target_length={pred.shape[1]}, channels={pred.shape[2]}")

## 📊 2. Interpreting Predictions

The prediction tensor has 18 channels:
- **Channels 0-2** -- Splice type (neither, acceptor, donor)
- **Channels 3-17** -- Tissue-specific usage (15 tissues)

In [ ]:
pred = np.array(result.prediction[0])  # First (only) sequence

# Splice type probabilities
neither_probs = pred[:, SpliceTransformerType.NEITHER.value]
acceptor_probs = pred[:, SpliceTransformerType.ACCEPTOR.value]
donor_probs = pred[:, SpliceTransformerType.DONOR.value]

print(f"Max acceptor probability: {acceptor_probs.max():.4f} at position {acceptor_probs.argmax()}")
print(f"Max donor probability: {donor_probs.max():.4f} at position {donor_probs.argmax()}")
print(f"Mean 'neither' probability: {neither_probs.mean():.4f}")

## 🎯 3. Finding High-Confidence Splice Sites

In [4]:
threshold = 0.5

acceptor_sites = np.where(acceptor_probs > threshold)[0]
donor_sites = np.where(donor_probs > threshold)[0]

print(f"High-confidence acceptor sites (>{threshold}): {len(acceptor_sites)}")
if len(acceptor_sites) > 0:
    print(f"  Positions: {acceptor_sites[:10]}..." if len(acceptor_sites) > 10 else f"  Positions: {acceptor_sites}")

print(f"High-confidence donor sites (>{threshold}): {len(donor_sites)}")
if len(donor_sites) > 0:
    print(f"  Positions: {donor_sites[:10]}..." if len(donor_sites) > 10 else f"  Positions: {donor_sites}")

High-confidence acceptor sites (>0.5): 0
High-confidence donor sites (>0.5): 0


## 🧪 4. Tissue-Specific Analysis

Channels 3-17 provide tissue-specific splice usage predictions.

In [ ]:
# Examine tissue-specific predictions
pred = np.array(result.prediction[0])
for tissue_name, channel_idx in SPLICE_TISSUE_CHANNEL_INDEX.items():
    if channel_idx is None:  # AVERAGE is handled separately via channel slice 3:
        continue
    tissue_pred = pred[:, channel_idx]
    print(f"{tissue_name:20s}: mean={tissue_pred.mean():.4f}, max={tissue_pred.max():.4f}")

## 💾 5. Export Results

Export predictions to files for downstream analysis.

In [ ]:
# Export predictions to NPY format
result.export("splice_transformer_predictions", export_path="./example_output", file_format="npy")
print("Predictions exported to ./example_output/splice_transformer_predictions.npy")